# Tome 9, Chapter 2: Tensors and Exactness (Python companion)

Tensor products of abelian groups computed from presentations, the exact sequence of a composite checked on finite groups, and product codes, the tensor product of two error-correcting codes.

*Before each solution, predict what the output will look like.*

## Exercises

**P2.1 (★) Tensor products of abelian groups by Smith normal form.** A finitely generated abelian group given by generators and relations is $M=\operatorname{coker}(A)$ for an integer matrix $A$ ($\Z^n\to\Z^m$), and its invariant factors are the diagonal of the Smith normal form of $A$ (Tome 2, Chapter 7). (a) Show, using right exactness (Corollary 2.13), that if $M=\operatorname{coker}A$ ($m\times n$) and $N=\operatorname{coker}B$ ($p\times q$), then $M\otimes N=\operatorname{coker}\,[A\otimes I_p\;\;I_m\otimes B]$, a matrix of size $mp\times(np+mq)$. (b) Implement it with `sympy` and compare with the formula of Example 2.11 on random groups. (c) Check by brute force on small finite groups that $|\operatorname{Hom}(M,N)|=|M\otimes N|$ for finite abelian $M,N$ (true because both are sums of $\Z/\gcd(m_i,n_j)$).

**P2.2 (★) The exact sequence of a composite.** For homomorphisms $f\colon A\to B$, $g\colon B\to C$ between finite abelian groups, Exercise 2.3 gives an exact sequence of six finite groups. In any exact sequence $0\to G_1\to\cdots\to G_k\to0$ of finite groups, $\prod_i|G_i|^{(-1)^i}=1$ (prove it by splitting into short exact sequences). Test this on random homomorphisms between groups $(\Z/N)^a$, computing kernels and images by brute force.

**P2.3 (★★) CS application: product codes.** A binary linear code $C\subseteq\mathbb F_2^n$ of dimension $k$ and minimum distance $d$ (the least weight of a non-zero codeword) is an $[n,k,d]$ code. The *product code* $C_1\otimes C_2\subseteq\mathbb F_2^{n_1}\otimes\mathbb F_2^{n_2}$, the $n_1\times n_2$ arrays all of whose columns lie in $C_1$ and rows in $C_2$, is the tensor product of the two subspaces; a generator matrix is the Kronecker product $G_1\otimes G_2$. (a) Build the Hamming $[7,4,3]$ code and its square, and check that it is a $[49,16,9]$ code by listing its $2^{16}$ codewords. (b) Prove that $d(C_1\otimes C_2)=d_1d_2$ in general. (c) Decode by correcting rows and columns alternately with the Hamming decoder, and estimate how often random patterns of $t$ errors are corrected, for $t=1,\dots,8$. Product codes of this kind protect CDs (cross-interleaved Reed–Solomon codes) and the data in QR codes and flash memory.

## Solutions

### P2.1

(a) By definition of $M=\operatorname{coker}A$, the sequence $\Z^n\xrightarrow A\Z^m\to M\to0$ is exact. Tensoring with $N$ (right exact) gives $N^n\to N^m\to M\otimes N\to0$, and $N^m=\operatorname{coker}(I_m\otimes B)$. So $M\otimes N$ is $\Z^{mp}$ modulo the images of $I_m\otimes B$ and of $A\otimes I_p$.

In [ ]:
import random, math, itertools
from sympy import Matrix, ZZ, eye, gcd
from sympy.matrices.normalforms import smith_normal_form
random.seed(2)

def invariants(A):
    """Invariant factors of coker(A), A an m x n integer matrix: torsion orders > 1 and the free rank."""
    m = A.shape[0]
    if A.shape[1] == 0:
        return [], m
    S = smith_normal_form(A, domain=ZZ)
    d = [abs(S[i, i]) for i in range(min(S.shape))]
    torsion = sorted(x for x in d if x > 1)
    free = m - sum(1 for x in d if x != 0)
    return torsion, free

def kron(A, B):
    return Matrix(A.shape[0] * B.shape[0], A.shape[1] * B.shape[1],
                  lambda i, j: A[i // B.shape[0], j // B.shape[1]] * B[i % B.shape[0], j % B.shape[1]])

def tensor(A, B):
    m, p = A.shape[0], B.shape[0]
    return invariants(kron(A, eye(p)).row_join(kron(eye(m), B)))

def formula(MA, MB):
    """Example 2.11: torsion and rank of the tensor product from the decompositions."""
    (tm, a), (tn, b) = MA, MB
    tors = [x for x in tm for _ in range(b)] + [x for x in tn for _ in range(a)]
    tors += [math.gcd(x, y) for x in tm for y in tn]
    return sorted(t for t in tors if t > 1), a * b

def canon(tors):
    """Write a finite abelian group canonically by its elementary divisors (prime powers)."""
    out = []
    for t in tors:
        n, p = t, 2
        while n > 1:
            if n % p == 0:
                q = 1
                while n % p == 0:
                    n //= p; q *= p
                out.append(q)
            p += 1
    return sorted(out)

ok = True
for trial in range(30):
    A = Matrix(random.randint(1, 3), random.randint(0, 3), lambda i, j: random.randint(-6, 6))
    B = Matrix(random.randint(1, 3), random.randint(0, 3), lambda i, j: random.randint(-6, 6))
    MA, MB = invariants(A), invariants(B)
    T = tensor(A, B)
    F = formula(MA, MB)
    ok &= (canon(T[0]), T[1]) == (canon(F[0]), F[1])
    if trial < 5:
        print("M =", MA, " N =", MB, "  M (x) N =", T)
print("all 30 agree with Example 2.11:", ok)

Each group is printed as (torsion invariant factors, free rank). The matrix computation and the formula agree once both are written in elementary divisors ($\Z/6\cong\Z/2\oplus\Z/3$, so the two lists of invariant factors may differ before that).

In [ ]:
def hom_count(m_list, n_list):
    """Brute-force count of homomorphisms between finite abelian groups prod Z/m_i -> prod Z/n_j:
    a homomorphism is the image of each generator e_i, an element x with m_i x = 0."""
    N = list(itertools.product(*[range(n) for n in n_list]))
    count = 1
    for m in m_list:
        count *= sum(1 for x in N if all((m * xi) % n == 0 for xi, n in zip(x, n_list)))
    return count

for M, N in [([4], [6]), ([2, 4], [4]), ([3, 9], [9, 3]), ([2, 2, 2], [4, 2]), ([12], [18])]:
    t = math.prod(math.gcd(a, b) for a in M for b in N)
    print(f"M = {M}, N = {N}:  |Hom(M,N)| = {hom_count(M, N)},  |M (x) N| = {t}")

Both are $\prod_{i,j}\gcd(m_i,n_j)$. The two groups are even isomorphic for finite abelian groups, but not *naturally*: one is covariant in $N$, the other contravariant in $M$.

### P2.2

*Proof of the product formula.* For a short exact sequence $0\to A\to B\to C\to0$ of finite groups, $|B|=|A||C|$. A long exact sequence $0\to G_1\xrightarrow{u_1}G_2\to\cdots\to G_k\to0$ breaks into the short exact sequences $0\to\operatorname{im}u_{i-1}\to G_i\to\operatorname{im}u_i\to0$, so $|G_i|=|\operatorname{im}u_{i-1}||\operatorname{im}u_i|$; in the alternating product every $|\operatorname{im}u_i|$ appears once with each sign.

In [ ]:
import numpy as np
rng = np.random.default_rng(22)

def elements(N, a):
    return [np.array(v) for v in itertools.product(range(N), repeat=a)]

def check(N, a, b, c):
    F = rng.integers(0, N, (b, a)); G = rng.integers(0, N, (c, b))
    A, B = elements(N, a), elements(N, b)
    f = lambda x: tuple(F @ x % N); g = lambda y: tuple(G @ np.array(y) % N)
    ker_f = sum(1 for x in A if not any(f(x)))
    ker_gf = sum(1 for x in A if not any(g(f(x))))
    ker_g = sum(1 for y in B if not any(g(tuple(y))))
    im_f = len({f(x) for x in A}); im_gf = len({g(f(x)) for x in A}); im_g = len({g(tuple(y)) for y in B})
    coker_f, coker_gf, coker_g = N**b // im_f, N**c // im_gf, N**c // im_g
    orders = [ker_f, ker_gf, ker_g, coker_f, coker_gf, coker_g]
    alt = (ker_f * ker_g * coker_gf) / (ker_gf * coker_f * coker_g)
    return orders, alt

for (N, a, b, c) in [(4, 2, 2, 2), (6, 3, 2, 2), (12, 2, 3, 2), (8, 3, 3, 1), (9, 2, 2, 3)]:
    orders, alt = check(N, a, b, c)
    print(f"Z/{N}: |ker f|, |ker gf|, |ker g|, |coker f|, |coker gf|, |coker g| = {orders};  alternating product = {alt}")

The alternating product is $1$ every time. Note what this says about the obvious guess: $|\ker(g\circ f)|$ is *not* $|\ker f|\,|\ker g|$ in general; the correction is exactly the connecting map $\ker g\to\operatorname{coker}f$ of the snake.

### P2.3

In [ ]:
G7 = np.array([[1,0,0,0,1,1,0],
               [0,1,0,0,1,0,1],
               [0,0,1,0,0,1,1],
               [0,0,0,1,1,1,1]])            # Hamming [7,4,3] in systematic form
H7 = np.array([[1,1,0,1,1,0,0],
               [1,0,1,1,0,1,0],
               [0,1,1,1,0,0,1]])
assert not (G7 @ H7.T % 2).any()

def min_distance(G):
    k = G.shape[0]
    best = G.shape[1]
    for bits in range(1, 2**k):
        msg = np.array([(bits >> i) & 1 for i in range(k)])
        best = min(best, int((msg @ G % 2).sum()))
    return best

G49 = np.kron(G7, G7) % 2
print("Hamming code: n, k, d =", G7.shape[1], G7.shape[0], min_distance(G7))
print("product code: n, k, d =", G49.shape[1], np.linalg.matrix_rank(G49), min_distance(G49))

(b) A non-zero array $X$ in $C_1\otimes C_2$ has a non-zero column, which lies in $C_1$ and so has at least $d_1$ non-zero entries; each of the $d_1$ rows through them is a non-zero word of $C_2$, with at least $d_2$ ones. So $\operatorname{wt}X\ge d_1d_2$, and $x\otimes y$ with $x,y$ of minimal weight has weight exactly $d_1d_2$. (The rank printed is over $\mathbb R$, which equals $16$ here as over $\mathbb F_2$; the minimum distance $9$ confirms that the $2^{16}$ messages give distinct codewords.)

In [ ]:
syndrome_table = {tuple(H7[:, j]): j for j in range(7)}

def hamming_correct(word):
    s = tuple(H7 @ word % 2)
    w = word.copy()
    if any(s):
        w[syndrome_table[s]] ^= 1
    return w

def decode_product(X, rounds=4):
    X = X.copy()
    for _ in range(rounds):
        X = np.array([hamming_correct(r) for r in X])          # rows
        X = np.array([hamming_correct(c) for c in X.T]).T      # columns
    return X

trials = 4000
for t in range(1, 9):
    good = 0
    for _ in range(trials):
        msg = rng.integers(0, 2, 16)
        X = (msg @ G49 % 2).reshape(7, 7)
        E = np.zeros(49, dtype=int); E[rng.choice(49, t, replace=False)] = 1
        good += np.array_equal(decode_product((X + E.reshape(7, 7)) % 2), X)
    print(f"{t} errors: corrected in {100 * good / trials:5.1f}% of trials")

Every pattern of one, two or three errors tried was corrected. With $4$ errors the simple row-and-column decoder fails on a small fraction of patterns (four errors at the corners of a rectangle fool every row and column decoder), although distance $9$ means that a maximum-likelihood decoder corrects all of them: decoding a product code well is harder than building it. Beyond $4$ errors many patterns are still corrected, which is why product codes do much better against random errors than their minimum distance promises.